# 🦌 WildEye — Robustness Evaluation

**Mini Hackathon #1: How Can Machines See What Matters?**

Both models score ~96% on a clean test set. This notebook answers the real question:

> **What happens when reality intervenes?**

We take the **same test set**, apply synthetic perturbations that simulate real camera-trap conditions, and measure how much each model's accuracy degrades. The model that degrades less is the more robust one — and that's the one we want in the field.

| Perturbation | Simulates |
|---|---|
| Night / IR | Camera switches to infrared after dark |
| Motion blur | Animal moving when shutter fires |
| Low light + noise | Dusk, dawn, high ISO |
| Fog | Mist, early morning, forest haze |
| Occlusion | Leaves / branches blocking the lens |
| Combined | All of the above at once (worst case) |

---

## 1. Mount Drive & Setup

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/deeplearn/hack1')
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q albumentations scikit-learn matplotlib seaborn tqdm pillow

In [ ]:
import json, random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
print(f'Device: {DEVICE}')

## 2. Reconstruct the Same Test Split

Must use the same `SEED` and split ratios as the training notebooks so we evaluate on the exact same images.

In [ ]:
# --- Config (must match training notebooks) ---
DATA_DIR   = '/content/data/raw-img'   # local copy (fast IO)
IMAGE_SIZE = 224
BATCH_SIZE = 64   # larger batch is fine for eval-only
NUM_WORKERS = 0
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

# Check data is available locally; if not, remind user
if not Path(DATA_DIR).exists():
    print('⚠️  Local data not found. Re-download from Kaggle:')
    print('  !kaggle datasets download -d alessiocorrado99/animals10 -p /content/ --force')
    print('  !unzip -q /content/animals10.zip -d /content/data/')
    print('  Then rename Italian folders to English (see notebook 02).')
else:
    print('Local data found ✓')

In [ ]:
!kaggle datasets download -d alessiocorrado99/animals10 -p /content/ --force
!unzip -q /content/animals10.zip -d /content/data/
  # Then rename Italian folders to English (see notebook 02).

In [ ]:
# # 确保 kaggle.json 还在（如果 Colab 重连过需要重新上传）
# import os
# if not os.path.exists('/root/.kaggle/kaggle.json'):
#     from google.colab import files
#     files.upload()  # 选 kaggle.json
#     !mkdir -p ~/.kaggle
#     !mv kaggle.json ~/.kaggle/kaggle.json
#     !chmod 600 ~/.kaggle/kaggle.json

# # # 直接下载到本地 /content/，完全不走 Drive
# # !kaggle datasets download -d alessiocorrado99/animals10 -p /content/ --force
# # !unzip -q /content/animals10.zip -d /content/data/
# # !rm /content/animals10.zip

# 重命名
from pathlib import Path
LOCAL_DATA = Path('/content/data/raw-img')
translate = {
    'cane': 'dog', 'cavallo': 'horse', 'elefante': 'elephant',
    'farfalla': 'butterfly', 'gallina': 'chicken', 'gatto': 'cat',
    'mucca': 'cow', 'pecora': 'sheep', 'ragno': 'spider',
    'scoiattolo': 'squirrel',
}
for it, en in translate.items():
    src, dst = LOCAL_DATA / it, LOCAL_DATA / en
    if src.exists():
        src.rename(dst)
        print(f'✓ {it} → {en}')

print('\nDone:')
for d in sorted(LOCAL_DATA.iterdir()):
    if d.is_dir():
        n = len(list(d.rglob('*.jpg')))
        print(f'  {d.name:15s}: {n}')

In [ ]:
def scan_folder(root):
    root = Path(root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    class_to_idx = {c: i for i, c in enumerate(classes)}
    samples, exts = [], {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    for cls in classes:
        for p in (root / cls).rglob('*'):
            if p.suffix.lower() in exts:
                samples.append((str(p), class_to_idx[cls]))
    return samples, class_to_idx

all_samples, class_to_idx = scan_folder(DATA_DIR)
idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES  = len(class_to_idx)

paths, labels = [s[0] for s in all_samples], [s[1] for s in all_samples]
_, temp_p, _, temp_l = train_test_split(
    paths, labels,
    test_size=VAL_SPLIT + TEST_SPLIT,
    stratify=labels, random_state=SEED,
)
val_ratio = VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT)
_, test_p, _, test_l = train_test_split(
    temp_p, temp_l, test_size=1 - val_ratio,
    stratify=temp_l, random_state=SEED,
)
test_samples = list(zip(test_p, test_l))
print(f'Test set: {len(test_samples)} images across {NUM_CLASSES} classes')
for idx, cnt in sorted(Counter([s[1] for s in test_samples]).items()):
    print(f'  {idx_to_class[idx]:15s}: {cnt}')

## 3. Perturbation Definitions

Six perturbation levels, each with **three severity levels** (mild / moderate / severe).
The `Combined` perturbation stacks all conditions at once — the worst-case real field scenario.

In [ ]:
def make_perturbation(name, severity):
    """Return an albumentations Compose for a given perturbation + severity.
    severity: 1=mild, 2=moderate, 3=severe
    """
    s = severity  # shorthand

    base = [A.Resize(IMAGE_SIZE, IMAGE_SIZE)]

    if name == 'clean':
        aug = []

    elif name == 'night_ir':
        # Grayscale + brightness drop to simulate IR night vision
        aug = [
            A.ToGray(p=1.0),
            A.RandomBrightnessContrast(
                brightness_limit=(-0.1*s*2, -0.1*s),
                contrast_limit=(-0.1, 0.1), p=1.0
            ),
        ]

    elif name == 'motion_blur':
        aug = [A.MotionBlur(blur_limit=(3 + s*4, 5 + s*6), p=1.0)]

    elif name == 'low_light':
        aug = [
            A.RandomBrightnessContrast(
                brightness_limit=(-0.2*s, -0.1*s),
                contrast_limit=(-0.2, 0.0), p=1.0
            ),
            A.GaussNoise(std_range=(0.03*s, 0.08*s), p=1.0),
        ]

    elif name == 'fog':
    # 用亮度+对比度模拟雾感，比 RandomFog 快 100x
        coef = 0.15 * s
        aug = [
            A.RandomBrightnessContrast(
                brightness_limit=(coef*0.5, coef),
                contrast_limit=(-coef, -coef*0.3),
                p=1.0
            ),
        ]

    elif name == 'occlusion':
        aug = [A.CoarseDropout(
            num_holes_range=(s*3, s*6),
            hole_height_range=(16*s, 24*s),
            hole_width_range=(16*s, 24*s),
            p=1.0
        )]

    elif name == 'combined':
        # Worst-case: stack everything at given severity
        coef = min(0.1 * s, 0.4)
        aug = [
            A.ToGray(p=0.8),
            A.MotionBlur(blur_limit=(3 + s*3, 5 + s*5), p=0.7),
            A.RandomBrightnessContrast(
                brightness_limit=(-0.15*s, -0.05*s),
                contrast_limit=(-0.15, 0.0), p=0.9
            ),
            A.GaussNoise(std_range=(0.02*s, 0.06*s), p=0.7),
            # A.RandomFog(fog_coef_range=(coef*0.7, coef), p=0.5),
            # A.RandomFog(fog_coef_lower=coef*0.7,
            #             fog_coef_upper=coef, p=0.5),
            A.CoarseDropout(
                num_holes_range=(s*2, s*4),
                hole_height_range=(12*s, 20*s),
                hole_width_range=(12*s, 20*s),
                p=0.6
            ),
        ]

    else:
        raise ValueError(f'Unknown perturbation: {name}')

    return A.Compose(base + aug + [
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


PERTURBATIONS = ['clean', 'night_ir', 'motion_blur', 'low_light', 'fog', 'occlusion', 'combined']
SEVERITIES    = [1, 2, 3]
SEVERITY_LABELS = {1: 'mild', 2: 'moderate', 3: 'severe'}

print('Perturbation definitions ready.')
print(f'Conditions: {PERTURBATIONS}')

## 4. Visualize Perturbations

One image × all perturbations × all severities. **Use this in your pitch slides.**

In [ ]:
def denormalize(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)

# Pick one sample per class for the visualization
sample_path = test_samples[0][0]
img_pil = Image.open(sample_path).convert('RGB')
img_np  = np.array(img_pil)

n_cols = len(PERTURBATIONS)
n_rows = len(SEVERITIES)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.8, n_rows * 2.8))

for row, sev in enumerate(SEVERITIES):
    for col, pert in enumerate(PERTURBATIONS):
        tf = make_perturbation(pert, sev)
        out = tf(image=img_np)['image']
        axes[row][col].imshow(denormalize(out))
        axes[row][col].axis('off')
        if row == 0:
            axes[row][col].set_title(pert.replace('_', '\n'), fontsize=9, fontweight='bold')
        if col == 0:
            axes[row][col].set_ylabel(SEVERITY_LABELS[sev], fontsize=9, rotation=0,
                                       labelpad=45, va='center')

plt.suptitle('Perturbation Grid — same image across all conditions and severities',
             fontsize=11, y=1.01)
plt.tight_layout()
vis_path = PROJECT_ROOT / 'results/figures/perturbation_grid.png'
plt.savefig(vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {vis_path}')

## 5. Load Both Models

In [ ]:
def build_model(num_classes):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


def load_checkpoint(ckpt_path, num_classes):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = build_model(num_classes)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(DEVICE)
    return model


models_dir = PROJECT_ROOT / 'models'

baseline_model = load_checkpoint(
    models_dir / 'baseline_no_aug_best.pth', NUM_CLASSES
)
domain_model = load_checkpoint(
    models_dir / 'domain_aug_best.pth', NUM_CLASSES
)

MODELS = {
    'Baseline (no aug)': baseline_model,
    'Domain Aug (ours)': domain_model,
}
print('Both models loaded ✓')

## 6. Evaluation Loop

For every perturbation × severity combination, evaluate both models and record accuracy + F1. This cell is the core of the notebook — takes ~5 minutes.

In [ ]:
# 把测试图片全部预加载到内存，避免 36 次重复的 Drive IO
print('Preloading test images into RAM...')
test_images_cache = {}
for path, label in tqdm(test_samples, desc='Loading'):
    test_images_cache[path] = np.array(Image.open(path).convert('RGB'))
print(f'Cached {len(test_images_cache)} images ✓')

In [ ]:
class PerturbedTestSet(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = test_images_cache[path]   # ← 从内存读，不走 Drive
        img = self.transform(image=img.copy())['image']  # copy 防止 albumentations 修改原数组
        return img, label

In [ ]:
import os

results = []
results_path = PROJECT_ROOT / 'results/robustness_results.json'

# 如果已有部分结果，先加载进来
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f'Resumed from checkpoint: {len(results)} results already done')

# 在评估 cell 最上面加这几行，清掉 fog 的旧结果
results = [r for r in results if r['perturbation'] != 'fog']
done = {(r['perturbation'], r['severity'], r['model']) for r in results}
print(f'After cleanup: {len(results)} results')

# 重新保存清理后的结果
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

# 记录已经完成的 (pert, sev, model) 组合
done = {(r['perturbation'], r['severity'], r['model']) for r in results}

conditions = [('clean', 1)]
for pert in PERTURBATIONS[1:]:
    for sev in SEVERITIES:
        conditions.append((pert, sev))

for pert, sev in tqdm(conditions, desc='Evaluating'):
    tf = make_perturbation(pert, sev)
    loader = DataLoader(
        PerturbedTestSet(test_samples, tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    )
    for model_name, model in MODELS.items():
        if (pert, sev, model_name) in done:
            continue   # 跳过已完成的
        acc, f1 = evaluate(model, loader)
        results.append({
            'perturbation': pert,
            'severity': sev,
            'model': model_name,
            'accuracy': acc,
            'f1_macro': f1,
        })
        # 每完成一个就立刻保存
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)

print(f'Evaluation complete. {len(results)} results saved → {results_path}')

## 7. Results Table

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

# Pivot for readability
pivot = df.pivot_table(
    index=['perturbation', 'severity'],
    columns='model',
    values='accuracy'
).round(4)

pivot['Δ (Domain - Baseline)'] = (
    pivot['Domain Aug (ours)'] - pivot['Baseline (no aug)']
).round(4)

print('Accuracy by perturbation & severity')
print('=' * 65)
print(pivot.to_string())
print('\n(Positive Δ = domain aug is more robust)')

## 8. 📊 Robustness Degradation Curves

**This is the pitch chart.** Each subplot shows one perturbation type. X-axis = severity (mild → severe). Lines = model accuracy. The model whose line stays higher is more robust.

In [2]:
COLORS = {
    'Baseline (no aug)': '#e74c3c',
    'Domain Aug (ours)': '#2ecc71',
}
PERTS_TO_PLOT = PERTURBATIONS[1:]  # exclude 'clean'
clean_accs = {
    m: df[(df.perturbation == 'clean') & (df.model == m)]['accuracy'].values[0]
    for m in MODELS
}

n_perts = len(PERTS_TO_PLOT)
ncols = 3
nrows = (n_perts + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4), sharey=False)
axes = axes.flat

for ax, pert in zip(axes, PERTS_TO_PLOT):
    sub = df[df.perturbation == pert]
    for model_name, color in COLORS.items():
        sub_m = sub[sub.model == model_name].sort_values('severity')
        # prepend severity=0 = clean accuracy as starting point
        x = [0] + sub_m['severity'].tolist()
        y = [clean_accs[model_name]] + sub_m['accuracy'].tolist()
        ax.plot(x, y, color=color, linewidth=2.5, marker='o',
                label=model_name)
        ax.fill_between(x, y, alpha=0.08, color=color)

    ax.set_title(pert.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Severity  (0=clean, 1=mild, 2=moderate, 3=severe)')
    ax.set_ylabel('Accuracy')
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(['clean', 'mild', 'mod.', 'severe'])
    ax.set_ylim(0, 1.05)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=8)
    ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

# Hide unused subplots
for ax in list(axes)[n_perts:]:
    ax.set_visible(False)

plt.suptitle(
    'Robustness Degradation: Baseline vs Domain Augmentation\n'
    'Green (domain aug) staying higher = more robust to real-world conditions',
    fontsize=12, y=1.01
)
plt.tight_layout()
deg_path = PROJECT_ROOT / 'results/figures/robustness_degradation.png'
plt.savefig(deg_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {deg_path}')

NameError: name 'PERTURBATIONS' is not defined

## 9. 📊 Robustness Summary Bar Chart

Average accuracy drop from clean → severe, per perturbation type. **Smaller drop = more robust.**

In [ ]:
summary = []
for pert in PERTS_TO_PLOT:
    for model_name in MODELS:
        severe = df[
            (df.perturbation == pert) &
            (df.severity == 3) &
            (df.model == model_name)
        ]['accuracy'].values[0]
        drop = clean_accs[model_name] - severe
        summary.append({
            'perturbation': pert.replace('_', ' ').title(),
            'model': model_name,
            'accuracy_drop': drop,
            'severe_acc': severe,
        })

sum_df = pd.DataFrame(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy at severe perturbation
perts  = sum_df['perturbation'].unique()
x      = np.arange(len(perts))
width  = 0.35
ax = axes[0]
for i, (model_name, color) in enumerate(COLORS.items()):
    vals = sum_df[sum_df.model == model_name]['severe_acc'].values
    bars = ax.bar(x + i*width - width/2, vals, width,
                  color=color, label=model_name, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x)
ax.set_xticklabels(perts, rotation=20, ha='right')
ax.set_ylabel('Accuracy at Severe Perturbation')
ax.set_title('Accuracy under worst-case conditions\n(higher = more robust)')
ax.set_ylim(0, 1.1)
ax.legend(); ax.spines[['top', 'right']].set_visible(False)

# Right: accuracy drop (clean → severe)
ax = axes[1]
for i, (model_name, color) in enumerate(COLORS.items()):
    vals = sum_df[sum_df.model == model_name]['accuracy_drop'].values
    bars = ax.bar(x + i*width - width/2, vals, width,
                  color=color, label=model_name, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x)
ax.set_xticklabels(perts, rotation=20, ha='right')
ax.set_ylabel('Accuracy Drop (clean → severe)')
ax.set_title('Accuracy degradation under worst-case conditions\n(lower = more robust)')
ax.legend(); ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Robustness Summary — Severe Perturbation Level', fontsize=12)
plt.tight_layout()
bar_path = PROJECT_ROOT / 'results/figures/robustness_summary_bar.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {bar_path}')

## 10. Final Summary Numbers

In [ ]:
print('=' * 60)
print('ROBUSTNESS EVALUATION SUMMARY')
print('=' * 60)

print(f'\nClean test accuracy:')
for m, acc in clean_accs.items():
    print(f'  {m:25s}: {acc:.4f}')

print(f'\nAverage accuracy at SEVERE perturbation:')
for model_name in MODELS:
    severe_accs = [
        r['accuracy'] for r in results
        if r['model'] == model_name and r['severity'] == 3
        and r['perturbation'] != 'clean'
    ]
    avg = np.mean(severe_accs)
    drop = clean_accs[model_name] - avg
    print(f'  {model_name:25s}: {avg:.4f}  (drop: -{drop:.4f})')

print(f'\nWorst-case (Combined, severe):')
for model_name in MODELS:
    acc = next(
        r['accuracy'] for r in results
        if r['model'] == model_name
        and r['perturbation'] == 'combined'
        and r['severity'] == 3
    )
    print(f'  {model_name:25s}: {acc:.4f}')

print('\n' + '=' * 60)
print('KEY FINDING FOR YOUR PITCH:')
base_combined = next(
    r['accuracy'] for r in results
    if r['model'] == 'Baseline (no aug)'
    and r['perturbation'] == 'combined' and r['severity'] == 3
)
aug_combined = next(
    r['accuracy'] for r in results
    if r['model'] == 'Domain Aug (ours)'
    and r['perturbation'] == 'combined' and r['severity'] == 3
)
print(f'Under worst-case field conditions:')
print(f'  Baseline accuracy  : {base_combined:.1%}')
print(f'  Domain aug accuracy: {aug_combined:.1%}')
print(f'  Robustness gain    : +{(aug_combined - base_combined):.1%}')
print('=' * 60)

## ✅ Done!

### Files produced
```
results/
├── robustness_results.json
└── figures/
    ├── perturbation_grid.png        ← show real conditions in pitch
    ├── robustness_degradation.png   ← THE main pitch chart
    └── robustness_summary_bar.png   ← supporting evidence
```

### Pitch narrative for Section 10 summary

1. *"On clean data, both models are about equal (~96%)."*
2. *"But camera traps don't give you clean data."* → show `perturbation_grid.png`
3. *"Under realistic conditions, the baseline degrades sharply."* → show `robustness_degradation.png`
4. *"Domain augmentation keeps accuracy X% higher under worst-case conditions."* → `robustness_summary_bar.png`
5. *"The cost? 0.4% on clean data. The benefit? Staying reliable when it matters."*

### Next
Deploy the domain aug model as a Streamlit app with live perturbation sliders — `04_streamlit_app.py`.

In [4]:
from google.colab import files
files.download('/content/drive/MyDrive/deeplearn/hack1/models/domain_aug_best.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from pathlib import Path

translate = {
    'cane': 'dog', 'cavallo': 'horse', 'elefante': 'elephant',
    'farfalla': 'butterfly', 'gallina': 'chicken', 'gatto': 'cat',
    'mucca': 'cow', 'pecora': 'sheep', 'ragno': 'spider',
    'scoiattolo': 'squirrel',
}

img_root = Path('/content/drive/MyDrive/deeplearn/hack1/data/raw/raw-img')
for it, en in translate.items():
    src, dst = img_root / it, img_root / en
    if src.exists():
        src.rename(dst)
        print(f'✓ {it} → {en}')
    elif dst.exists():
        print(f'  {en} already renamed')
    else:
        print(f'✗ {it} not found')

✓ cane → dog
✓ cavallo → horse
✓ elefante → elephant
✓ farfalla → butterfly
✓ gallina → chicken
✓ gatto → cat
✓ mucca → cow
✓ pecora → sheep
✓ ragno → spider
✓ scoiattolo → squirrel


In [10]:
from PIL import Image
import numpy as np
import albumentations as A
from google.colab import files

# 随便找一张本地图片，不需要 test_samples
path = '/content/drive/MyDrive/deeplearn/hack1/data/raw/raw-img/dog/OIP-_-AtcUnGMN6ht4EYKmgkXgHaE8.jpeg'
# 或者任意一张：
# import glob
# path = glob.glob('/content/data/raw-img/dog/*.jpg')[0]

img_np = np.array(Image.open(path).convert('RGB'))

tf = A.Compose([
    A.Resize(224, 224),
    A.MotionBlur(blur_limit=(10, 15), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=(-0.5, -0.3), p=1.0),
    A.GaussNoise(std_range=(0.1, 0.15), p=1.0),
])
aug = tf(image=img_np)['image']

out = Image.fromarray(aug)
out.save('/content/drive/MyDrive/deeplearn/hack1/test_blurry.jpg')
files.download('/content/test_blurry.jpg')